# Machine Learning Project
# Trashing Bad Cards
# Kylle Waldie

## imports

In [1]:
import cv2
import numpy as np
import os
import shutil
import argparse
from tqdm import tqdm

## Start of Code

### Parameters

In [2]:
BACKGROUND_TEXTURE_THRESHOLD = 12   # Std dev of background region. Lower = stricter.
MIN_CARD_AREA_RATIO = 0.25          # Card must fill at least this fraction of the image.
MAX_CARDS_DETECTED = 1              # Images with more than this many cards = bad.
MIN_CARD_ASPECT = 0.55              # Min aspect ratio for a card-like rectangle (w/h).
MAX_CARD_ASPECT = 0.90              # Max aspect ratio for a card-like rectangle (w/h).

### Paths

In [3]:
RAW_DIR    = r"C:\Users\wldky\OneDrive - Montana Tech\Spring-2026\CSCI 447\ML Project\dataset\raw"
OUTPUT_DIR = r"C:\Users\wldky\OneDrive - Montana Tech\Spring-2026\CSCI 447\ML Project\dataset\sorted"

### Helper Functions

In [4]:
def find_card_contours(img):
    """Find rectangular contours that look like cards."""
    h, w = img.shape[:2]
    image_area = h * w

    # Converts cards to grey scale and then blurs them.
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # Finds places where pixel brightness changes sharply, which is where edges are.
    edges = cv2.Canny(blurred, 30, 100)

    # Dilation makes the detected edges slightly thicker
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    edges = cv2.dilate(edges, kernel, iterations=2)

    # RETR_EXTERNAL means we only want the outermost outlines, not holes inside shapes. Traces the points around the card.
    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    card_contours = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < image_area * 0.05: # Makes sure the card takes up most of the space.
            continue
        peri = cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, 0.02 * peri, True) # Simplifys the contour into straight lines and checks corners.
        if not (4 <= len(approx) <= 6):
            continue
        x, y, cw, ch = cv2.boundingRect(cnt)
        aspect = min(cw, ch) / max(cw, ch) # Makes sure the card is taller than it is wider.
        if not (MIN_CARD_ASPECT <= aspect <= MAX_CARD_ASPECT):
            continue
        if area / image_area < MIN_CARD_AREA_RATIO: # Makes sure the card takes up at least 25% of the image.
            continue
        card_contours.append(cnt)

    return card_contours

In [5]:
def measure_background_complexity(img, card_contours):
    """
    Measure how busy the background is outside the card area.
    High std dev = busy/textured background (bad).
    """
    h, w = img.shape[:2]
    mask = np.ones((h, w), dtype=np.uint8) * 255 # Creates a white mask the same size as the image, then paints the card area black. This leaves only the background region white.
    for cnt in card_contours:
        cv2.drawContours(mask, [cnt], -1, 0, thickness=cv2.FILLED)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    background_pixels = gray[mask == 255] # Calculates the standard deviation. 
    if len(background_pixels) < 100:
        return 0.0
    return float(np.std(background_pixels)) # If this score is above BACKGROUND_TEXTURE_THRESHOLD = 18, it's flagged as bad.

In [6]:
def classify_image(image_path):
    """Returns ('good', reason) or ('bad', reason) for a given image."""
    img = cv2.imread(image_path)
    if img is None:
        return 'bad', 'Could not read image'

    card_contours = find_card_contours(img)
    num_cards = len(card_contours)

    if num_cards == 0:
        return 'bad', 'No card-like rectangle detected'
    if num_cards > MAX_CARDS_DETECTED:
        return 'bad', f'Multiple cards detected ({num_cards})'

    bg_complexity = measure_background_complexity(img, card_contours)
    if bg_complexity > BACKGROUND_TEXTURE_THRESHOLD:
        return 'bad', f'Busy background (texture score: {bg_complexity:.1f})'

    return 'good', f'Single card, clean background (texture score: {bg_complexity:.1f})'

## Main Function

In [7]:
def sort_images():
    # Find all PSA subfolders
    psa_folders = sorted([
        f for f in os.listdir(RAW_DIR)
        if os.path.isdir(os.path.join(RAW_DIR, f)) and f.startswith('PSA_')
    ])

    if not psa_folders:
        print(f"No PSA_* subfolders found in:\n  {RAW_DIR}")
        return

    # Create output folder structure
    for folder in psa_folders:
        os.makedirs(os.path.join(OUTPUT_DIR, 'good', folder), exist_ok=True)
        os.makedirs(os.path.join(OUTPUT_DIR, 'bad',  folder), exist_ok=True)

    total_good = 0
    total_bad  = 0
    log        = []

    for psa_folder in psa_folders:
        folder_path = os.path.join(RAW_DIR, psa_folder)
        jpgs = [f for f in os.listdir(folder_path) if f.lower().endswith('.jpg')]

        grade_good = 0
        grade_bad  = 0

        print(f"Processing {psa_folder} ({len(jpgs)} images)...")

        for filename in jpgs:
            src = os.path.join(folder_path, filename)
            label, reason = classify_image(src)

            dst = os.path.join(OUTPUT_DIR, label, psa_folder, filename)
            shutil.copy2(src, dst)

            if label == 'good':
                grade_good += 1
                total_good += 1
            else:
                grade_bad += 1
                total_bad += 1

            log.append((psa_folder, filename, label, reason))

        print(f"Kept:    {grade_good}")
        print(f"Removed: {grade_bad}")
        print()

    # Final summary
    total = total_good + total_bad
    print(f"{'='*40}")
    print(f"All grades complete!")
    print(f"Total kept:    {total_good} / {total}")
    print(f"Total removed: {total_bad} / {total}")
    print(f"{'='*40}")

    # Save log
    log_path = os.path.join(OUTPUT_DIR, 'sort_log.txt')
    with open(log_path, 'w', encoding='utf-8') as f:
        f.write(f"Card Sort Log\n{'='*60}\n")
        f.write(f"Good: {total_good} | Bad: {total_bad} | Total: {total}\n\n")
        current_folder = None
        for psa_folder, fname, label, reason in log:
            if psa_folder != current_folder:
                f.write(f"\n── {psa_folder} ──\n")
                current_folder = psa_folder
            f.write(f"  [{label.upper()}] {fname} — {reason}\n")

    print(f"Log saved to: {log_path}")

In [8]:
sort_images()

Processing PSA_08 (1 images)...
Kept:    0
Removed: 1

Processing PSA_1 (135 images)...
Kept:    48
Removed: 87

Processing PSA_10 (558 images)...
Kept:    312
Removed: 246

Processing PSA_13 (1 images)...
Kept:    0
Removed: 1

Processing PSA_15 (1 images)...
Kept:    0
Removed: 1

Processing PSA_19 (1 images)...
Kept:    0
Removed: 1

Processing PSA_2 (50 images)...
Kept:    28
Removed: 22

Processing PSA_20 (2 images)...
Kept:    0
Removed: 2

Processing PSA_23 (1 images)...
Kept:    0
Removed: 1

Processing PSA_3 (1112 images)...
Kept:    374
Removed: 738

Processing PSA_4 (1104 images)...
Kept:    410
Removed: 694

Processing PSA_40 (1 images)...
Kept:    0
Removed: 1

Processing PSA_5 (2714 images)...
Kept:    872
Removed: 1842

Processing PSA_6 (1086 images)...
Kept:    484
Removed: 602

Processing PSA_7 (5468 images)...
Kept:    1758
Removed: 3710

Processing PSA_70 (1 images)...
Kept:    0
Removed: 1

Processing PSA_8 (5944 images)...
Kept:    1894
Removed: 4050

Processing PS